# NeuralEF — Comparison with PIEFS (GPU)

> **Before running:** Runtime → Change runtime type → **T4 GPU**

Implements **NeuralEF** (Deng et al., ICML 2022) from scratch on GPU, evaluated on all PIEFS
benchmark datasets with the **same experimental protocol** (same MLP architecture, K, seeds, evaluation).

## Algorithm (Deng et al. 2022)

NeuralEF learns K eigenfunctions of an RBF kernel k(x,x') = exp(−‖x−x'‖²/(2σ²)).

**Loss:**  
Given batch Φ ∈ ℝ^{B×K} and kernel matrix K_B ∈ ℝ^{B×B}:  
`S = Φᵀ K_B Φ / B²`  (kernel Gram matrix estimate)  
`L = ‖S − I‖²_F`  (same structural form as PIEFS gram loss, but in kernel space)

After training: classify with LogisticRegression on φ_1,...,φ_K features.

**Metrics:** acc, auc, f1, mcc, gram_error (‖C_feature−I‖²_F), kernel_gram_error, eigenvalues, training curves.

Results saved to `Google Drive/PIEFS_NeuralEF/` and downloaded as ZIP.

**Datasets:** Two Moons, Circles, HTRU2, MNIST-bin, MNIST-10, CIFAR-bin, CIFAR-10 features

In [ ]:
!pip install -q torch torchvision scikit-learn numpy tqdm

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}  ({gpu.total_memory / 1e9:.1f} GB)')
    DEVICE = torch.device('cuda')
else:
    print('WARNING: No GPU found — running on CPU (much slower).')
    print('Go to Runtime → Change runtime type → T4 GPU')
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/PIEFS_NeuralEF'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive output dir: {DRIVE_DIR}')

In [ ]:
# ── Data setup (no GitHub required) ───────────────────────────────────────
import os
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

# HTRU2: download from UCI archive (auto if not cached)
HTRU2_CSV = DATA_DIR / 'HTRU_2.csv'
if not HTRU2_CSV.exists():
    import urllib.request, zipfile, io
    print('Downloading HTRU2 from UCI...')
    url = 'https://archive.ics.uci.edu/static/public/372/htru2.zip'
    with urllib.request.urlopen(url) as r:
        zdata = r.read()
    with zipfile.ZipFile(io.BytesIO(zdata)) as zf:
        name = [n for n in zf.namelist() if n.endswith('.csv')][0]
        zf.extract(name, DATA_DIR)
        extracted = DATA_DIR / name
        extracted.rename(HTRU2_CSV)
    print(f'HTRU2 saved → {HTRU2_CSV}')
else:
    print(f'HTRU2 already cached: {HTRU2_CSV}')

print('Data setup complete.')

In [ ]:
from __future__ import annotations
import json, math, shutil, time, warnings, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score
)
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

SEEDS            = [0, 1, 2, 3, 4]
BATCH_SIZE       = 512 if DEVICE.type == 'cuda' else 256
LR               = 1e-3
STEPS_BINARY     = 30_000
STEPS_MULTICLASS = 50_000
K_BINARY         = 6
K_MULTICLASS     = 10
HIDDEN_DIMS      = [64, 64, 64]  # same as PIEFS basis network

DATASETS_TO_RUN = [
    'two_moon',
    'circles',
    'htru2',
    'mnist_binary',
    'mnist_mc',
    'cifar10_binary',
    'cifar10_mc',
]

OUT_DIR    = Path('/content/results_neuralef')
OUT_DIR.mkdir(exist_ok=True)
DRIVE_PATH = Path(DRIVE_DIR)

print(f'Batch size: {BATCH_SIZE},  Device: {DEVICE}')

In [ ]:
# ── Dataset loaders (self-contained, no PIEFS source needed) ─────────────

import numpy as np
import torch
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def _standardize(X_tr, X_te):
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr).astype(np.float32), sc.transform(X_te).astype(np.float32)


def load_two_moon(seed):
    X, y = make_moons(n_samples=10_000, noise=0.1, random_state=0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
    X_tr, X_te = _standardize(X_tr, X_te)
    return X_tr, y_tr, X_te, y_te


def load_circles(seed):
    X, y = make_circles(n_samples=10_000, noise=0.05, random_state=0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
    X_tr, X_te = _standardize(X_tr, X_te)
    return X_tr, y_tr, X_te, y_te


def load_htru2(seed):
    data = np.loadtxt(str(HTRU2_CSV), delimiter=',')
    X, y = data[:, :8].astype(np.float32), data[:, 8].astype(int)
    rng = np.random.default_rng(42)
    idx = rng.permutation(len(X))
    X, y = X[idx], y[idx]
    n_train = int(len(X) * 0.7)
    X_tr, y_tr = X[:n_train], y[:n_train]
    X_te, y_te = X[n_train:], y[n_train:]
    X_tr, X_te = _standardize(X_tr, X_te)
    return X_tr, y_tr, X_te, y_te


def _load_torchvision(name, task, binary_classes=(0, 1), val_fraction=0.1, seed=42):
    import torchvision.datasets as tvd
    import torchvision.transforms as T
    to_tensor = T.ToTensor()
    root = str(DATA_DIR / name)
    if name == 'mnist':
        tr = tvd.MNIST(root, train=True,  download=True, transform=to_tensor)
        te = tvd.MNIST(root, train=False, download=True, transform=to_tensor)
    elif name == 'cifar10':
        tr = tvd.CIFAR10(root, train=True,  download=True, transform=to_tensor)
        te = tvd.CIFAR10(root, train=False, download=True, transform=to_tensor)
    else:
        raise ValueError(name)

    def to_numpy(ds):
        loader = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=False)
        xs, ys = [], []
        for xb, yb in loader:
            xs.append(xb.flatten(1).float().numpy())
            ys.append(yb.numpy())
        return np.vstack(xs), np.concatenate(ys)

    X_tr_all, y_tr_all = to_numpy(tr)
    X_te,     y_te     = to_numpy(te)

    if task == 'binary':
        mask_tr = (y_tr_all == binary_classes[0]) | (y_tr_all == binary_classes[1])
        mask_te = (y_te     == binary_classes[0]) | (y_te     == binary_classes[1])
        X_tr_all = X_tr_all[mask_tr]
        y_tr_all = (y_tr_all[mask_tr] == binary_classes[1]).astype(int)
        X_te     = X_te[mask_te]
        y_te     = (y_te[mask_te]     == binary_classes[1]).astype(int)

    rng2 = np.random.default_rng(seed)
    perm = rng2.permutation(len(X_tr_all))
    n_val = int(len(X_tr_all) * val_fraction)
    train_idx = perm[n_val:]
    X_tr = X_tr_all[train_idx]
    y_tr = y_tr_all[train_idx]

    sc = StandardScaler().fit(X_tr)
    X_tr = sc.transform(X_tr).astype(np.float32)
    X_te = sc.transform(X_te).astype(np.float32)
    return X_tr, y_tr, X_te, y_te


def load_mnist_binary(seed):
    return _load_torchvision('mnist', 'binary')

def load_mnist_mc(seed):
    return _load_torchvision('mnist', 'multiclass')

def load_cifar10_binary(seed):
    return _load_torchvision('cifar10', 'binary')

def load_cifar10_mc(seed):
    return _load_torchvision('cifar10', 'multiclass')


LOADERS = {
    'two_moon':       (load_two_moon,       'binary'),
    'circles':        (load_circles,        'binary'),
    'htru2':          (load_htru2,          'binary'),
    'mnist_binary':   (load_mnist_binary,   'binary'),
    'mnist_mc':       (load_mnist_mc,       'multiclass'),
    'cifar10_binary': (load_cifar10_binary, 'binary'),
    'cifar10_mc':     (load_cifar10_mc,     'multiclass'),
}
print('Loaders registered:', list(LOADERS))

In [ ]:
# ── NeuralEF model ─────────────────────────────────────────────────────────

class NeuralEFNet(nn.Module):
    """Shared MLP outputting K eigenfunctions. Same arch as PIEFS BasisNet."""
    def __init__(self, input_dim: int, K: int, hidden_dims: list[int]):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers.append(nn.Linear(prev, K))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=0.5)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)  # (B, K)


def median_bandwidth(X: np.ndarray, subsample: int = 3000) -> float:
    """RBF bandwidth σ via median heuristic (runs on CPU)."""
    idx   = np.random.choice(len(X), min(subsample, len(X)), replace=False)
    X_sub = torch.tensor(X[idx], dtype=torch.float32)  # CPU is fine here
    dists = torch.cdist(X_sub, X_sub, p=2).view(-1)
    sigma = float(torch.median(dists[dists > 0]).item())
    return sigma / math.sqrt(2)


def rbf_kernel_gpu(x: torch.Tensor, sigma: float) -> torch.Tensor:
    """RBF kernel matrix on GPU. x: (B, d) → returns (B, B)."""
    dists_sq = torch.cdist(x, x, p=2) ** 2
    return torch.exp(-dists_sq / (2.0 * sigma ** 2))


def neuralef_loss(phi: torch.Tensor, K_mat: torch.Tensor):
    """‖S − I‖²_F where S = ΦᵀK_BΦ / B²."""
    B, K = phi.shape
    S = phi.t() @ K_mat @ phi / (B ** 2)
    I = torch.eye(K, device=phi.device)
    loss = ((S - I) ** 2).sum()
    return loss, S

print('NeuralEF model defined')

In [ ]:
# ── Training function (GPU-accelerated) ────────────────────────────────────

def train_neuralef(X_tr, y_tr, task, seed, total_steps, K, log_every=5000):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = NeuralEFNet(X_tr.shape[1], K, HIDDEN_DIMS).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # Bandwidth estimated on CPU (no large memory needed)
    sigma = median_bandwidth(X_tr)

    # Pin data to CPU; will move batches to GPU in loop
    X_t  = torch.tensor(X_tr, dtype=torch.float32)
    ds   = TensorDataset(X_t, torch.zeros(len(X_t)))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                        drop_last=True, num_workers=0, pin_memory=(DEVICE.type=='cuda'))
    loader_iter = iter(loader)
    train_log   = []
    t0 = time.time()

    for step in range(1, total_steps + 1):
        try:
            x_batch, _ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x_batch, _ = next(loader_iter)

        x_batch = x_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        phi   = model(x_batch)               # (B, K) on GPU
        K_mat = rbf_kernel_gpu(x_batch, sigma)  # (B, B) on GPU
        loss, S = neuralef_loss(phi, K_mat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        if step % log_every == 0 or step == 1:
            with torch.no_grad():
                ev  = S.diag().cpu().tolist()
                kge = float(loss.item())
            train_log.append({'step': step, 'kernel_gram_error': round(kge, 6),
                              'eigenvalues': [round(e, 4) for e in ev]})
            if step % (log_every * 2) == 0:
                elapsed = time.time() - t0
                print(f'    step {step:6d}/{total_steps}  loss={kge:.4f}  '
                      f'ev={[round(e,2) for e in ev[:3]]}  ({elapsed:.0f}s)')

    return {'model': model.cpu(), 'sigma': sigma,
            'train_log': train_log, 'fit_time': round(time.time()-t0, 2)}

print('Training function ready')

In [ ]:
# ── Evaluation (on CPU, sklearn) ───────────────────────────────────────────

@torch.no_grad()
def extract_features(model, X, batch_size=2048):
    """Extract features in CPU mini-batches (avoids OOM for large datasets)."""
    model.eval()
    parts = []
    for i in range(0, len(X), batch_size):
        x = torch.tensor(X[i:i+batch_size], dtype=torch.float32)
        parts.append(model(x).numpy())
    return np.vstack(parts)


def evaluate_neuralef(model, X_tr, y_tr, X_te, y_te, task, seed):
    phi_tr = extract_features(model, X_tr)
    phi_te = extract_features(model, X_te)

    multi = 'multinomial' if task == 'multiclass' else 'auto'
    clf = LogisticRegression(solver='lbfgs', max_iter=2000, C=1.0,
                              multi_class=multi, random_state=seed)
    t0 = time.time()
    clf.fit(phi_tr, y_tr)
    clf_time = time.time() - t0

    y_pred  = clf.predict(phi_te)
    y_proba = clf.predict_proba(phi_te)
    avg = 'binary' if task == 'binary' else 'macro'

    acc = float(accuracy_score(y_te, y_pred))
    f1  = float(f1_score(y_te, y_pred, average=avg, zero_division=0))
    mcc = float(matthews_corrcoef(y_te, y_pred))
    pre = float(precision_score(y_te, y_pred, average=avg, zero_division=0))
    rec = float(recall_score(y_te, y_pred, average=avg, zero_division=0))
    try:
        auc = float(roc_auc_score(y_te, y_proba[:,1] if task=='binary' else y_proba,
                                   multi_class='ovr' if task=='multiclass' else 'raise',
                                   average='macro'))
    except Exception:
        auc = float('nan')

    # gram_error = ‖C_feature − I‖²_F   (same metric as PIEFS val_gram_error)
    C = phi_te.T @ phi_te / len(phi_te)
    gram_error = float(np.sum((C - np.eye(phi_te.shape[1])) ** 2))

    return dict(acc=round(acc,6), auc=round(auc,6), f1_macro=round(f1,6),
                mcc=round(mcc,6), precision_macro=round(pre,6), recall_macro=round(rec,6),
                gram_error=round(gram_error,6), clf_time=round(clf_time,3))

print('Evaluation function ready')

In [ ]:
# ── Main experiment loop ───────────────────────────────────────────────────

all_results = {}
resume_path = OUT_DIR / 'neuralef_results.json'
if resume_path.exists():
    with open(resume_path) as f:
        all_results = json.load(f)
    print(f'Resumed from {resume_path}')

for ds_name in DATASETS_TO_RUN:
    loader_fn, task = LOADERS[ds_name]
    total_steps = STEPS_MULTICLASS if task == 'multiclass' else STEPS_BINARY
    K = K_MULTICLASS if task == 'multiclass' else K_BINARY

    print(f'\n{"="*70}')
    print(f'DATASET: {ds_name}  task={task}  K={K}  steps={total_steps}  device={DEVICE}')
    print(f'{"="*70}')

    all_results.setdefault(ds_name, {})

    for seed in tqdm(SEEDS, desc=ds_name):
        if str(seed) in all_results[ds_name]:
            print(f'  seed={seed}: already done, skipping')
            continue

        try:
            X_tr, y_tr, X_te, y_te = loader_fn(seed)
        except Exception as e:
            print(f'  [SKIP] seed={seed}: {e}')
            continue

        print(f'  seed={seed}: train={len(X_tr)}, test={len(X_te)}, d={X_tr.shape[1]}')

        res   = train_neuralef(X_tr, y_tr, task, seed, total_steps, K)
        model = res['model']  # already on CPU after training

        metrics = evaluate_neuralef(model, X_tr, y_tr, X_te, y_te, task, seed)

        all_results[ds_name][str(seed)] = {
            **metrics,
            'fit_time':               res['fit_time'],
            'sigma':                  round(res['sigma'], 4),
            'total_steps':            total_steps,
            'K':                      K,
            'n_train':                len(X_tr),
            'n_test':                 len(X_te),
            'input_dim':              X_tr.shape[1],
            'train_log':              res['train_log'],
            'final_eigenvalues':      res['train_log'][-1]['eigenvalues'] if res['train_log'] else [],
            'final_kernel_gram_error': res['train_log'][-1]['kernel_gram_error'] if res['train_log'] else None,
        }

        m = metrics
        print(f'    acc={m["acc"]:.4f}  auc={m["auc"]:.4f}  f1={m["f1_macro"]:.4f}  '
              f'mcc={m["mcc"]:.4f}  gram_err={m["gram_error"]:.4f}  t={res["fit_time"]:.0f}s')

        with open(resume_path, 'w') as f:
            json.dump(all_results, f, indent=2)
        shutil.copy(resume_path, DRIVE_PATH / 'neuralef_results.json')
        torch.cuda.empty_cache()

print('\nAll done.')

In [ ]:
# ── Results table ──────────────────────────────────────────────────────────
import statistics

METRICS_SHOW = ['acc', 'auc', 'f1_macro', 'mcc', 'gram_error', 'final_kernel_gram_error', 'fit_time']
datasets = list(all_results.keys())

print('=== NeuralEF — mean±std over seeds ===')
hdr = f'{"Dataset":<20}' + ''.join(f'{m[:14]:>17}' for m in METRICS_SHOW)
print(hdr); print('-' * len(hdr))

for ds in datasets:
    row = f'{ds:<20}'
    for metric in METRICS_SHOW:
        vals = [all_results[ds].get(str(s), {}).get(metric) for s in SEEDS]
        vals = [v for v in vals if v is not None and v == v]
        if vals:
            mv = statistics.mean(vals)
            sv = statistics.stdev(vals) if len(vals) > 1 else 0.0
            sc = 100 if metric in ('acc', 'f1_macro') else 1
            row += f'{mv*sc:>9.2f}±{sv*sc:.2f}'
        else:
            row += f'{"—":>17}'
    print(row)

print('\n=== Final eigenvalue estimates (mean over seeds) ===')
print(f'{"Dataset":<20}' + ''.join(f'{"ev_"+str(i+1):>8}' for i in range(6)))
print('-' * 68)
for ds in datasets:
    evs = [all_results[ds].get(str(s), {}).get('final_eigenvalues', []) for s in SEEDS]
    evs = [e for e in evs if e]
    if evs:
        K6 = min(len(evs[0]), 6)
        means = [statistics.mean([e[k] for e in evs if k < len(e)]) for k in range(K6)]
        print(f'{ds:<20}' + ''.join(f'{v:>8.3f}' for v in means))

In [ ]:
# ── Training curves plot ───────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    n = len(datasets)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, ds in zip(axes, datasets):
        for s in SEEDS:
            log = all_results.get(ds, {}).get(str(s), {}).get('train_log', [])
            if log:
                ax.plot([r['step'] for r in log],
                        [r['kernel_gram_error'] for r in log], alpha=0.6, lw=1)
        ax.set_title(ds, fontsize=9); ax.set_xlabel('step')
        ax.set_ylabel('‖S−I‖²_F'); ax.set_yscale('log'); ax.grid(True, alpha=0.3)
    plt.suptitle('NeuralEF — kernel gram error', y=1.02)
    plt.tight_layout()
    fig_path = OUT_DIR / 'neuralef_curves.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    shutil.copy(fig_path, DRIVE_PATH / 'neuralef_curves.png')
    plt.show()
except Exception as e:
    print(f'Plot skipped: {e}')

In [ ]:
# ── Save to Drive + download ZIP ───────────────────────────────────────────
from google.colab import files

with open(resume_path, 'w') as f:
    json.dump(all_results, f, indent=2)
shutil.copy(resume_path, DRIVE_PATH / 'neuralef_results.json')

zip_path = '/content/piefs_neuralef.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(OUT_DIR))

shutil.copy(zip_path, DRIVE_PATH / 'piefs_neuralef.zip')
print(f'ZIP saved to Drive: {DRIVE_PATH}/piefs_neuralef.zip')
files.download(zip_path)